In [0]:
path = "/Volumes/workspace/default/avis_mode/Womens Clothing E-Commerce Reviews.csv"

df_raw = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(path))

df_raw.printSchema()
print(df_raw.count())
display(df_raw.limit(5))

root
 |-- _c0: string (nullable = true)
 |-- Clothing ID: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Review Text: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Recommended IND: string (nullable = true)
 |-- Positive Feedback Count: string (nullable = true)
 |-- Division Name: string (nullable = true)
 |-- Department Name: string (nullable = true)
 |-- Class Name: string (nullable = true)

23486


_c0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,null,Absolutely wonderful - silky and sexy and comfortable,4,1,0,Initmates,Intimate,Intimates
1,1080,34,null,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c",3,0,0,General,Dresses,Dresses
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",5,1,0,General Petite,Bottoms,Pants
4,847,47,Flattering shirt,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,5,1,6,General,Tops,Blouses


In [0]:
from pyspark.sql import functions as F

# 1) Couche BRUTE : on renomme les colonnes (sans espaces) et on sauvegarde en table
renames = {
    "_c0": "id_avis", "Clothing ID": "clothing_id", "Age": "age",
    "Title": "title", "Review Text": "review_text", "Rating": "rating",
    "Recommended IND": "recommended",
    "Positive Feedback Count": "positive_feedback_count",
    "Division Name": "division_name", "Department Name": "department_name",
    "Class Name": "class_name",
}
df_bronze = df_raw
for old, new in renames.items():
    df_bronze = df_bronze.withColumnRenamed(old, new)

df_bronze.write.mode("overwrite").saveAsTable("workspace.default.avis_bronze")

# 2) Couche NETTOYÉE : types corrects, texte nettoyé, lignes invalides retirées
int_cols = ["id_avis", "clothing_id", "age", "rating", "recommended", "positive_feedback_count"]
df_silver = spark.table("workspace.default.avis_bronze")
for c in int_cols:
    df_silver = df_silver.withColumn(c, F.col(c).cast("int"))

df_silver = (df_silver
    .withColumn("title", F.trim("title"))
    .withColumn("review_text", F.trim("review_text"))
    .filter(F.col("review_text").isNotNull() & (F.length("review_text") > 0))
    .filter(F.col("rating").between(1, 5))
    .dropDuplicates(["clothing_id", "age", "review_text"])
    .withColumn("review_length", F.length("review_text"))
    .withColumn("word_count", F.size(F.split("review_text", r"\s+")))
)

df_silver.write.mode("overwrite").saveAsTable("workspace.default.avis_silver")

print("brute  :", df_bronze.count())
print("nettoyée:", df_silver.count())
display(df_silver.limit(5))

brute  : 23486
nettoyée: 22639


id_avis,clothing_id,age,title,review_text,rating,recommended,positive_feedback_count,division_name,department_name,class_name,review_length,word_count
0,767,33,null,Absolutely wonderful - silky and sexy and comfortable,4,1,0,Initmates,Intimate,Intimates,53,8
1,1080,34,null,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,1,4,General,Dresses,Dresses,303,62
2,1077,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c",3,0,0,General,Dresses,Dresses,500,98
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",5,1,0,General Petite,Bottoms,Pants,124,22
4,847,47,Flattering shirt,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,5,1,6,General,Tops,Blouses,192,36


In [0]:
# Couche AGRÉGÉE n°1 : statistiques par catégorie de vêtement
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.avis_gold_categories AS
SELECT department_name, class_name,
       COUNT(*)                        AS nb_avis,
       ROUND(AVG(rating), 2)           AS note_moyenne,
       ROUND(AVG(recommended) * 100, 1) AS pct_recommande,
       ROUND(AVG(word_count), 1)       AS mots_moyens
FROM workspace.default.avis_silver
WHERE class_name IS NOT NULL
GROUP BY department_name, class_name
""")

# Couche AGRÉGÉE n°2 : statistiques par tranche d'âge
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.avis_gold_ages AS
SELECT CASE WHEN age < 25 THEN '18-24'
            WHEN age < 35 THEN '25-34'
            WHEN age < 45 THEN '35-44'
            WHEN age < 55 THEN '45-54'
            ELSE '55+' END            AS tranche_age,
       COUNT(*)                        AS nb_avis,
       ROUND(AVG(rating), 2)           AS note_moyenne,
       ROUND(AVG(recommended) * 100, 1) AS pct_recommande
FROM workspace.default.avis_silver
GROUP BY 1
ORDER BY 1
""")

display(spark.table("workspace.default.avis_gold_categories").orderBy(F.desc("nb_avis")))
display(spark.table("workspace.default.avis_gold_ages"))

department_name,class_name,nb_avis,note_moyenne,pct_recommande,mots_moyens
Dresses,Dresses,6144,4.14,80.5,64.7
Tops,Knits,4626,4.15,81.2,54.7
Tops,Blouses,2983,4.14,80.7,60.7
Tops,Sweaters,1380,4.17,79.7,59.9
Bottoms,Pants,1350,4.26,83.3,62.1
Bottoms,Jeans,1103,4.35,87.9,59.5
Tops,Fine gauge,1059,4.24,83.3,57.4
Bottoms,Skirts,903,4.23,84.2,61.9
Jackets,Jackets,683,4.29,84.3,61.7
Intimate,Lounge,669,4.3,85.8,55.8


tranche_age,nb_avis,note_moyenne,pct_recommande
18-24,867,4.33,86.5
25-34,4962,4.12,79.4
35-44,7431,4.17,81.3
45-54,4944,4.16,81.8
55+,4435,4.28,84.8


In [0]:
from pyspark.sql import functions as F

stop = ["the","and","to","it","is","of","in","for","this","that","was","but","on",
        "with","my","have","are","so","me","be","as","not","very","too","just",
        "would","they","at","if","or","an","you","from","than","them","when","had",
        "were","she","her","can","after","all","more","also","did","been","will",
        "really","what","there","out","up","like","one","wear","got","its","don",
        "bit","little","but","only","how","other","only","which","off","some"]

def top_mots(df, label, n=20):
    return (df
        .select(F.explode(F.split(
            F.regexp_replace(F.lower("review_text"), r"[^a-z\s]", " "), r"\s+")).alias("mot"))
        .filter((F.length("mot") > 2) & (~F.col("mot").isin(stop)))
        .groupBy("mot").count()
        .orderBy(F.desc("count")).limit(n)
        .withColumn("groupe", F.lit(label)))

silver = spark.table("workspace.default.avis_silver")
neg = top_mots(silver.filter(F.col("rating") <= 2), "negatif")
pos = top_mots(silver.filter(F.col("rating") == 5), "positif")

mots = neg.unionByName(pos)
mots.write.mode("overwrite").saveAsTable("workspace.default.avis_gold_mots")

display(neg)
display(pos)

mot,count,groupe
dress,1121,negatif
top,830,negatif
fabric,712,negatif
fit,686,negatif
size,677,negatif
back,615,negatif
small,517,negatif
look,516,negatif
ordered,511,negatif
love,458,negatif


mot,count,groupe
love,6208,positif
dress,5720,positif
size,4898,positif
great,4101,positif
fit,4024,positif
top,3715,positif
perfect,2963,positif
color,2595,positif
small,2353,positif
flattering,2328,positif
